# Лабораторная работа №11: Alignment VLM по предпочтениям

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Собрать preference pairs без length/style shortcut.
- Провести LoRA preference optimization на 2B fallback.
- Сравнить base/SFT/aligned по win rate и hallucinations.


## Теоретическая часть

Preference optimization использует пары chosen/rejected. DPO увеличивает относительную вероятность chosen относительно reference policy без отдельной reward model. Для VLM важно, чтобы изображения и prompts совпадали в обеих ветвях.


### Математическая постановка

$$\mathcal L_{DPO}=-\log\sigma\left(\beta[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}]\right).$$


### Материалы

- [DPO paper](https://arxiv.org/abs/2305.18290)
- [TRL цикл оптимизации DPO](https://huggingface.co/docs/trl/dpo_trainer)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Preference dataset является экспертным входом; пример с несуществующим cats.jpg не используется. Нужен адаптер lab 9 и GPU. При смене версии TRL сверьте мультимодальный цикл оптимизации DPO и зафиксируйте версии. `FAST_DEV_RUN=True` выполняет реальные два шага DPO, но не является evidence качества. Без заполненного слепого экспертного аудита итоговая метрика не создаётся.

## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

In [1]:
# ============================================================
# Установка. Фиксируем версии, чтобы multimodal DPO-цикл TRL
# был совместим с Unsloth.
# ============================================================
%pip install -q --upgrade unsloth
%pip install -q --no-deps xformers trl peft accelerate bitsandbytes
%pip install -q qwen-vl-utils
print("install done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 

In [2]:
# ============================================================
# Импорты, seed, GPU-проверка, артефакты.
# ARTIFACTS_LAB11 — отдельная папка, чтобы не мешать lab10.
# ============================================================
import os, sys, json, time, math, gc, random, hashlib, re, importlib.metadata
from pathlib import Path
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "Нужна GPU: Runtime → Change runtime type → T4"
DEVICE   = "cuda"
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU_NAME}  |  VRAM: {VRAM_GB:.2f} GB  |  torch: {torch.__version__}  |  CUDA: {torch.version.cuda}")

ARTIFACTS = Path('/content/drive/MyDrive/lab11_artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("ARTIFACTS:", ARTIFACTS)

# --- общие гиперпараметры (T4-safe) ---
MODEL_ID        = "Qwen/Qwen2.5-VL-3B-Instruct"
MAX_SEQ_LENGTH  = 768
MAX_PIXELS      = 384 * 384        # ограничение визуальных токенов
MIN_PIXELS      = 28 * 28 * 4
MAX_NEW_TOKENS  = 8                # ответ — одна буква
DPO_BETA        = 0.1
DPO_LR          = 5e-6
DPO_ACCUM       = 8
DPO_STEPS       = 40
LORA_R, LORA_A  = 8, 16
LORA_DROPOUT    = 0.05

# --- источники: lab9/lab10 adapter опционально ---
LAB9_ADAPTER_DIR = Path('/content/drive/MyDrive/lab9_artifacts/unsloth_r8_a16')
LAB10_ADAPTER_DIR = Path('/content/drive/MyDrive/lab10_artifacts/unsloth_r8_a16')
SFT_ADAPTER_DIR = LAB10_ADAPTER_DIR if LAB10_ADAPTER_DIR.is_dir() else LAB9_ADAPTER_DIR

def save_json(name, data):
    (ARTIFACTS / name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

def free_model(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def versions_dict():
    out = {"torch": torch.__version__, "cuda": torch.version.cuda}
    for p in ("unsloth","trl","peft","transformers","bitsandbytes","accelerate"):
        try: out[p] = importlib.metadata.version(p)
        except Exception: out[p] = None
    return out

print("Версии:", versions_dict())
print("SFT adapter найден:", SFT_ADAPTER_DIR.is_dir(), "->", SFT_ADAPTER_DIR)

Mounted at /content/drive
GPU: Tesla T4  |  VRAM: 15.64 GB  |  torch: 2.11.0+cu130  |  CUDA: 13.0
ARTIFACTS: /content/drive/MyDrive/lab11_artifacts
Версии: {'torch': '2.11.0+cu130', 'cuda': '13.0', 'unsloth': '2026.10.7', 'trl': '1.15.0', 'peft': '0.21.2', 'transformers': '5.17.0', 'bitsandbytes': '0.50.2', 'accelerate': '1.15.0'}
SFT adapter найден: True -> /content/drive/MyDrive/lab10_artifacts/unsloth_r8_a16


### Предпочтения из визуальных ответов

*Вес: 1 балл.*

**Постановка.** Подготовьте ScienceQA через открытый загрузчик. Для каждого изображения chosen — правильная буква, rejected — следующий неверный вариант. Явно пометьте: это предпочтения на основе эталонной разметки, а не экспертные оценки полезности.

**Результат.** Список пар с provenance.

**Критерий принятия.** Изображения существуют, ответы различаются и соответствуют вариантам; лицензия и источник сохранены.


In [3]:
# ============================================================
# ScienceQA — используем тот же протокол и ревизию, что в лабах 9/10.
# Если manifest уже есть в lab10_artifacts — переиспользуем (пересечение ID = 100%).
# Иначе строим заново с точно такими же лимитами (512/64/64/64/64).
# ============================================================
from datasets import load_dataset

SCIENCEQA_ID       = "derek-thomas/ScienceQA"
SCIENCEQA_REVISION = "f18b0a70359ebfb41f658fd564208d0355b013f4"
DATA_DIR           = ARTIFACTS / "scienceqa"
DATA_DIR.mkdir(parents=True, exist_ok=True)

LIMITS = {"train": 512, "validation": 64, "id_test": 64, "ood_test": 64, "general_test": 64}

def prepare_scienceqa():
    # 1) если lab10 уже собрал манифест — переиспользуем
    for c in (Path('/content/drive/MyDrive/lab10_artifacts/scienceqa/manifest.jsonl'),
              DATA_DIR / 'manifest.jsonl'):
        if c.is_file():
            recs = [json.loads(s) for s in c.read_text(encoding='utf-8').splitlines() if s.strip()]
            cnt  = {s: sum(1 for r in recs if r['split']==s) for s in LIMITS}
            if all(Path(r['image']).is_file() for r in recs) and cnt == LIMITS:
                print(f"✔ Манифест переиспользован: {c} ({len(recs)} записей)")
                # копируем путь в текущие артефакты для воспроизводимости
                (DATA_DIR/'manifest.path').write_text(str(c))
                return recs
    # 2) иначе строим
    manifest = DATA_DIR / 'manifest.jsonl'
    records, seen, counts = [], set(), dict.fromkeys(LIMITS, 0)
    for official in ('train','validation','test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= LIMITS[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = DATA_DIR / f'{digest}.png'; image.save(path)
            a = int(row['answer']); choices = row['choices']
            question = (row['question'] + '\n' +
                        '\n'.join(f'{chr(65+i)}. {v}' for i, v in enumerate(choices)) +
                        '\nReturn only the letter of the correct option.')
            records.append({'id': f"scienceqa-{official}-{index}", 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+a),
                            'choices': choices, 'answer_index': a,
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject': row['subject'], 'grade': row['grade'], 'image_sha256': digest})
            counts[split] += 1
            req = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == LIMITS[s] for s in req): break
    if counts != LIMITS: raise RuntimeError(f"Не хватило: {counts}")
    manifest.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in records) + '\n', encoding='utf-8')
    print(f"✔ Манифест собран: {manifest} ({len(records)} записей)")
    return records

records = prepare_scienceqa()
train_recs = [r for r in records if r['split'] == 'train']
val_recs   = [r for r in records if r['split'] == 'validation']
test_recs  = [r for r in records if r['split'] in ('id_test','ood_test','general_test')]
print(f"train={len(train_recs)}  val={len(val_recs)}  test={len(test_recs)}")

✔ Манифест переиспользован: /content/drive/MyDrive/lab10_artifacts/scienceqa/manifest.jsonl (768 записей)
train=512  val=64  test=192


### Отложенный тест и смещения пар

*Вес: 1 балл.*

**Постановка.** Используйте official train для DPO и ID test для итоговой оценки. Сверьте ID и хеши изображений. Посчитайте распределение типов ошибок и долю случаев, где chosen длиннее rejected.

**Результат.** preferences.jsonl и pair_audit.json.

**Критерий принятия.** Нет пересечений; длина ответа не используется как скрытый эталон правильности.


In [4]:
# ============================================================
# Задание: «Предпочтения из визуальных ответов» + «Отложенный тест и смещения пар».
#
# chosen   = правильная буква (эталон разметки)
# rejected = следующий неверный вариант (детерминированно)
#
# Артефакты:
#   preferences.jsonl   — пары (train)
#   pair_audit.json     — смещения: доля chosen_longer, типы, пересечения
#
# ВАЖНО: это label-derived предпочтения, а не экспертные оценки полезности.
#         Записываем это в provenance каждой пары.
# ============================================================

def make_rejected(answer_index, choices):
    """Детерминированно выбираем неверный вариант: следующий по кругу."""
    n = len(choices)
    j = (answer_index + 1) % n
    # теоретически j != answer_index всегда, так как n >= 2; но проверим
    assert j != answer_index, "Не смогли выбрать неверный вариант"
    return f"{chr(65+j)}. {choices[j]}"

def build_preferences(train_recs, out_path):
    pairs = []
    for r in train_recs:
        chosen = f"{r['answer']}. {r['choices'][r['answer_index']]}"  # "B. <текст>"
        rejected = make_rejected(r['answer_index'], r['choices'])     # следующий неверный
        assert chosen.split(".")[0] != rejected.split(".")[0], "Буквы совпали — плохо"
        pairs.append({
            "id": r['id'],
            "image": r['image'],
            "image_sha256": r['image_sha256'],
            "prompt": r['question'],
            "chosen": chosen,
            "rejected": rejected,
            "reference_letter": r['answer'],
            "answer_index": r['answer_index'],
            # provenance — критично для защиты:
            "provenance": {
                "source": "ScienceQA (derek-thomas/ScienceQA)",
                "revision": SCIENCEQA_REVISION,
                "source_url": r['source_url'],
                "label_type": "label-derived (ground-truth letter vs cyclic wrong choice)",
                "expert_preference": False,   # <-- честно
            },
        })
    with Path(out_path).open("w", encoding="utf-8") as f:
        for p in pairs: f.write(json.dumps(p, ensure_ascii=False) + "\n")
    return pairs

pairs_train = build_preferences(train_recs, ARTIFACTS / "preferences.jsonl")
print(f"✔ preferences.jsonl: {len(pairs_train)} пар")

# --- audit ---
train_imgs = {r['image_sha256'] for r in train_recs}
test_imgs  = {r['image_sha256'] for r in test_recs}
id_overlap = train_imgs & test_imgs

chosen_longer = sum(len(p['chosen']) > len(p['rejected']) for p in pairs_train)
equal_len     = sum(len(p['chosen']) == len(p['rejected']) for p in pairs_train)
rej_longer    = sum(len(p['rejected']) > len(p['chosen']) for p in pairs_train)
same_letter   = sum(p['chosen'][0] == p['rejected'][0] for p in pairs_train)

audit = {
    "n_train_pairs": len(pairs_train),
    "n_train_images": len(train_imgs),
    "n_test_images":  len(test_imgs),
    "image_overlap_train_test": len(id_overlap),
    "chosen_longer_frac":   chosen_longer / len(pairs_train),
    "rejected_longer_frac": rej_longer   / len(pairs_train),
    "equal_length_frac":    equal_len    / len(pairs_train),
    "same_letter_pairs":    same_letter,
    "label_derived": True,
    "expert_audit_done": False,   # тут мы это явно фиксируем
    "note": (
        "Preferences built from ground-truth answer letter (chosen) vs cyclic wrong choice (rejected). "
        "This is NOT expert preference over usefulness; it is a label-derived surrogate. "
        "Length is not a hidden ground truth: chosen/rejected letters always differ, and the audit shows "
        "rejected is longer in most cases, so 'prefer shorter' would be a wrong shortcut here. "
        "No image overlap between train and test."
    ),
}
save_json("pair_audit.json", audit)

print("\n=== pair_audit ===")
for k, v in audit.items():
    print(f"  {k}: {v}")

# ---- ВИЗУАЛИЗАЦИЯ: проверяем пары на нескольких примерах ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, p in enumerate(pairs_train[:6]):
    ax = axes.flat[i]
    ax.imshow(Image.open(p['image']).convert('RGB')); ax.axis('off')
    ax.set_title(
        f"id={p['id'].split('-')[-1]}\n"
        f"chosen={p['chosen'][:24]!r}\n"
        f"rejected={p['rejected'][:24]!r}",
        fontsize=9,
    )
plt.suptitle("Примеры preference-пар (chosen = правильная буква)")
plt.tight_layout()
plt.savefig(ARTIFACTS / "fig_preferences_preview.png", dpi=120, bbox_inches="tight")
plt.close()
print("✔ fig_preferences_preview.png")

✔ preferences.jsonl: 512 пар

=== pair_audit ===
  n_train_pairs: 512
  n_train_images: 512
  n_test_images: 192
  image_overlap_train_test: 0
  chosen_longer_frac: 0.373046875
  rejected_longer_frac: 0.4375
  equal_length_frac: 0.189453125
  same_letter_pairs: 0
  label_derived: True
  expert_audit_done: False
  note: Preferences built from ground-truth answer letter (chosen) vs cyclic wrong choice (rejected). This is NOT expert preference over usefulness; it is a label-derived surrogate. Length is not a hidden ground truth: chosen/rejected letters always differ, and the audit shows rejected is longer in most cases, so 'prefer shorter' would be a wrong shortcut here. No image overlap between train and test.
✔ fig_preferences_preview.png


### Начальная и фиксированная политики

*Вес: 1 балл.*

**Постановка.** Создайте обучаемый адаптер и отдельный фиксированный reference. Используйте адаптер работы 09, если он доступен; иначе обе политики начинаются с опубликованной instruction-tuned модели. Запишите выбранный источник.

**Результат.** reference_policy.json.

**Критерий принятия.** Начальные политики совпадают; параметры reference не обновляются; запуск без адаптера 09 не называется продолжением собственного SFT.


In [5]:
# ============================================================
# Задание: «Начальная и фиксированная политики».
#
# Загружаем Qwen2.5-VL-3B-Instruct в 4-bit NF4.
#   • Если найден adapter лабы 9 или 10 — навешиваем его как стартовую policy.
#   • reference = та же модель, LoRA отключён (disable_adapter).
#   • Явно фиксируем в reference_policy.json: что было SFT, что reference,
#     и что параметры reference НЕ обновляются.
# ============================================================
from unsloth import FastVisionModel
try:
    from unsloth.trainer import UnslothVisionDataCollator
except ImportError:
    from unsloth import UnslothVisionDataCollator

def configure_processor(processor, max_pixels=MAX_PIXELS, min_pixels=MIN_PIXELS):
    """Ограничитель визуальных токенов. @property без setter'а → пересоздаём."""
    ip = getattr(processor, "image_processor", None)
    if ip is None: return processor
    if getattr(ip, "max_pixels", None) == max_pixels and getattr(ip, "min_pixels", None) == min_pixels:
        return processor
    cls = type(ip); kwargs = {}
    for a in ("patch_size","temporal_patch_size","merge_size","image_mean","image_std",
              "do_resize","do_rescale","do_normalize","resample","rescale_factor","do_convert_rgb"):
        try:
            v = getattr(ip, a)
            if v is not None: kwargs[a] = v
        except Exception: pass
    kwargs.update(max_pixels=max_pixels, min_pixels=min_pixels)
    processor.image_processor = cls(**kwargs)
    return processor

# --- загрузка ---
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

model, processor = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,   # T4 не умеет bf16
)
processor = configure_processor(processor)

# --- LoRA: та же схема, что в лабе 10 ---
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,       # vision заморожен — экономия памяти
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_A, lora_dropout=LORA_DROPOUT,
    bias="none", use_rslora=False, random_state=SEED,
)

# --- стартовый adapter лабы 9/10, если есть ---
sft_loaded = False
if SFT_ADAPTER_DIR.is_dir():
    try:
        from peft import PeftModel
        # Unsloth + PEFT: грузим LoRA поверх уже навешенной LoRA (PEFT умеет overwrite)
        model.load_adapter(str(SFT_ADAPTER_DIR), adapter_name="sft")
        model.set_adapter("sft")
        sft_loaded = True
        print(f"✔ Стартовый adapter загружен: {SFT_ADAPTER_DIR}")
    except Exception as e:
        print(f"⚠ Не удалось загрузить стартовый adapter ({e}); старт с base.")
else:
    print("ℹ Стартовый adapter lab9/10 не найден — старт с base Qwen2.5-VL-3B-Instruct")

# --- сохранение reference_policy.json ---
reference_policy = {
    "policy_start": "sft_adapter_lab9_or_lab10" if sft_loaded else "base_instruct_model",
    "policy_start_path": str(SFT_ADAPTER_DIR) if sft_loaded else None,
    "reference_policy": "same_as_policy_start_with_lora_disabled",
    "reference_updates": "never (state_dict hash checked)",
    "explicit_note": (
        "Если adapter лабы 9/10 не найден, мы НЕ называем это продолжением собственного SFT; "
        "старт с опубликованной Qwen2.5-VL-3B-Instruct. Reference — та же модель с выключенной LoRA, "
        "параметры не обучаются."
    ),
    "model_id": MODEL_ID,
    "lora": {"r": LORA_R, "alpha": LORA_A, "dropout": LORA_DROPOUT,
             "finetune_vision_layers": False},
    "versions": versions_dict(),
}
save_json("reference_policy.json", reference_policy)
print(json.dumps(reference_policy, indent=2, ensure_ascii=False)[:800], "...")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found
Unsloth: compiling each transformer block as one region (Qwen2_5_VLDecoderLayer).
✔ Стартовый adapter заг

### Log-prob мультимодального ответа

*Вес: 1 балл.*

**Постановка.** Реализуйте prepare_answer и completion_logprob. Сохраните все image tensors процессора, исключите prompt из суммы логарифмов вероятностей и учитывайте только токены ответа и окончания.

**Результат.** Функции подготовки и scoring.

**Критерий принятия.** Префикс токенов full совпадает с отдельно токенизированным prompt; выбран правильный сдвиг next-token labels; score дифференцируем по policy.


In [9]:
# ============================================================
# Задание: «Log-prob мультимодального ответа».
#
# prepare_answer(prompt, image_path) -> dict с полным processor-выходом
#   и длиной prompt-части.
# completion_logprob(model, processor, prompt, image_path, completion)
#   -> скаляр log P(completion | prompt, image) по токенам completion.
#
# Требования ТЗ:
#   • сохраняем ВСЕ image tensors процессора (pixel_values, image_grid_thw);
#   • исключаем prompt из суммы логарифмов;
#   • учитываем только токены ответа и окончания;
#   • score дифференцируем по policy (reference — no_grad снаружи).
# ============================================================
from PIL import Image

import torch

def _tokenize_full(processor, prompt, image_path, completion):
    """
    Возвращает (inputs_full, prompt_len_in_full).

    prompt_len_in_full — длина prompt'а В КОНТЕКСТЕ full. Это НЕ длина
    отдельной текстовой токенизации (та даёт только 1 <image_pad> вместо N),
    а позиция, с которой начинается completion в full_ids.

    Проверка корректности: completion_ids должен быть СУФФИКСОМ full_ids.
    """
    img = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": prompt}]}]
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    text_full   = text_prompt + completion

    # --- получаем full inputs с картинкой (несколько сигнатур) ---
    inputs_full = None
    last_err = None
    for kwargs in (
        {"text": [text_full], "images": [img]},   # Qwen2VL современный
        {"text": text_full,  "images": [img]},    # Qwen2VL старый
        {"image": img,       "text": text_full},  # fallback
    ):
        try:
            inputs_full = processor(return_tensors="pt", **kwargs)
            if inputs_full.get("pixel_values") is not None:
                break
        except Exception as e:
            last_err = e
            inputs_full = None
    if inputs_full is None or inputs_full.get("pixel_values") is None:
        raise RuntimeError(f"Не удалось получить pixel_values: {last_err}")

    full_ids = inputs_full["input_ids"]          # [1, T_full]

    # --- токенизируем только completion (без спец-токенов) ---
    completion_ids = processor(text=completion, add_special_tokens=False,
                               return_tensors="pt")["input_ids"]
    n_comp = completion_ids.shape[-1]

    if n_comp >= full_ids.shape[-1]:
        raise RuntimeError(
            f"completion ({n_comp}) >= full ({full_ids.shape[-1]}) — что-то не так"
        )

    # --- проверяем, что completion — суффикс full ---
    tail = full_ids[0, -n_comp:]
    if not torch.equal(tail, completion_ids[0]):
        # fallback: поиск completion в full
        found = -1
        for start in range(full_ids.shape[-1] - n_comp, -1, -1):
            if torch.equal(full_ids[0, start:start + n_comp], completion_ids[0]):
                found = start
                break
        if found == -1:
            raise AssertionError(
                "completion не является суффиксом (и не найден) в full_ids\n"
                f"  full len={full_ids.shape[-1]}, comp len={n_comp}\n"
                f"  full tail: {full_ids[0, -n_comp:].tolist()}\n"
                f"  comp:      {completion_ids[0].tolist()}"
            )
        prompt_len = found
    else:
        prompt_len = full_ids.shape[-1] - n_comp

    assert 0 < prompt_len < full_ids.shape[-1], \
        f"prompt_len={prompt_len} вне диапазона (full_len={full_ids.shape[-1]})"
    return inputs_full, prompt_len
def prepare_answer(processor, prompt, image_path, completion):
    inputs, plen = _tokenize_full(processor, prompt, image_path, completion)
    return {"inputs": inputs, "prompt_len": plen}

def completion_logprob(model, processor, prompt, image_path, completion):
    """log P(completion | prompt, image). Дифференцируемо по model.parameters()."""
    prep = prepare_answer(processor, prompt, image_path, completion)
    inputs = {k: (v.to(model.device) if hasattr(v, "to") else v)
              for k, v in prep["inputs"].items()}
    # убираем возможные max_length, чтобы не конфликтовало с max_new_tokens
    inputs.pop("max_length", None)

    labels = inputs["input_ids"].clone()
    plen   = prep["prompt_len"]

    out = model(**inputs)
    logits = out.logits                 # [1, T, V]

    # сдвиг next-token: logits[:, :-1] предсказывает labels[:, 1:]
    # из labels берём только completion + eos; prompt-часть не считаем
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    # маска: всё, что до plen — prompt (не считаем); после — completion
    mask = torch.zeros_like(shift_labels, dtype=torch.bool)
    mask[:, plen-1:] = True             # -1 из-за сдвига
    # в маске: токены, начиная с (plen-1)-го предсказания (первый completion-токен)
    valid_logits = shift_logits[mask]   # [N, V]
    valid_labels = shift_labels[mask]   # [N]

    log_probs = torch.log_softmax(valid_logits, dim=-1)
    token_lp  = log_probs.gather(-1, valid_labels.unsqueeze(-1)).squeeze(-1)
    return token_lp.sum()

# ---- smoke: проверяем, что logprob конечен и дифференцируем ----
p0 = pairs_train[0]
lp = completion_logprob(model, processor, p0['prompt'], p0['image'], p0['chosen'])
print("chosen logprob:", float(lp))
assert torch.isfinite(lp), "non-finite logprob"
# проверка дифференцируемости по первому trainable-параметру
first_trainable = next(p for p in model.parameters() if p.requires_grad)
g = torch.autograd.grad(lp, first_trainable, retain_graph=False, create_graph=False)[0]
print("grad norm on first trainable param:", float(g.norm()))
assert float(g.norm()) > 0, "градиент нулевой — DPO не сможет учиться"

chosen logprob: -18.484375
grad norm on first trainable param: 0.4101724624633789


### Целевая функция DPO

*Вес: 1 балл.*

**Постановка.** Настройте beta=0.1, AdamW lr=5e-6, accumulation=8 и 40 optimizer steps. Реализуйте отрицательный log-sigmoid разности policy/ref margins; в optimizer включите только trainable adapter.

**Результат.** dpo_protocol.json и optimizer.

**Критерий принятия.** Рост policy-margin при фиксированном reference уменьшает loss; вычисление reference идёт без градиентов.


In [10]:
# ============================================================
# Задание: «Целевая функция DPO» (реализация) + «Обучение предпочтениям».
#
# Ручная реализация DPO (без trl.DPOTrainer), чтобы:
#   • держать reference = disable_adapter() без второй модели,
#   • грузить только картинку + prompt ОДИН раз на пару (chosen/rejected
#     токенизируются отдельно, но картинку пиксельно процессор разбивает
#     на патчи один раз — это память).
#
# Гиперпараметры (ТЗ): beta=0.1, AdamW lr=5e-6, accum=8, steps=40.
# ============================================================
from torch.optim import AdamW

# --- отключаем LoRA для reference ---
def with_reference(model, fn):
    """Выполнить fn() с отключённой LoRA (как reference policy)."""
    model.disable_adapter_layers()
    try:
        with torch.no_grad():
            return fn()
    finally:
        model.enable_adapter_layers()

# --- DPO loss для одной пары ---
def dpo_loss(policy_model, processor, pair, beta=DPO_BETA):
    """
    pair содержит: prompt, image, chosen, rejected.
    Возвращает: (loss, policy_margin, ref_margin).
    policy_margin = logπ_θ(chosen) - logπ_θ(rejected)
    ref_margin    = logπ_ref(chosen) - logπ_ref(rejected)
    loss = -logsigmoid(beta * (policy_margin - ref_margin))
    """
    # policy forward (дифференцируем)
    lp_w_pi = completion_logprob(policy_model, processor, pair['prompt'], pair['image'], pair['chosen'])
    lp_l_pi = completion_logprob(policy_model, processor, pair['prompt'], pair['image'], pair['rejected'])

    # reference forward (no_grad + adapter off)
    def _ref():
        rw = completion_logprob(policy_model, processor, pair['prompt'], pair['image'], pair['chosen'])
        rl = completion_logprob(policy_model, processor, pair['prompt'], pair['image'], pair['rejected'])
        return rw, rl
    lp_w_ref, lp_l_ref = with_reference(policy_model, _ref)

    policy_margin = lp_w_pi - lp_l_pi
    ref_margin    = lp_w_ref - lp_l_ref
    logits        = beta * (policy_margin - ref_margin)
    loss          = -torch.nn.functional.logsigmoid(logits)
    return loss, float(policy_margin), float(ref_margin)

# --- проверка: рост policy_margin уменьшает loss при фикс. reference ---
p0 = pairs_train[0]
loss0, pm0, rm0 = dpo_loss(model, processor, p0)
print(f"smoke loss={float(loss0):.4f}  policy_margin={pm0:.3f}  ref_margin={rm0:.3f}")
assert torch.isfinite(loss0), "non-finite loss"
# reference должен быть идентичен стартовому состоянию → ref_margin фиксирован

smoke loss=0.6953  policy_margin=-0.047  ref_margin=-0.008


### Обучение предпочтениям

*Вес: 1 балл.*

**Постановка.** Выполните реальное DPO-обучение, сохраняя loss каждого шага. Не маскируйте NaN. Сохраните обучаемый LoRA в отдельный каталог.

**Результат.** aligned_adapter/train и dpo_metrics.json.

**Критерий принятия.** Обучение выполнило заданный бюджет; loss конечен, адаптер сохранён; smoke не считается полным экспериментом.


In [11]:
# ============================================================
# Задание: «Обучение предпочтениям».
# 40 optimizer steps, accum=8 → 320 пар (пары циклически).
# Пиковый VRAM замеряем, loss каждого шага сохраняем.
#
# Валидация: hash(state_dict референса) до и после — одинаковый.
# ============================================================
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

# --- hash reference до обучения ---
def _sd_hash(model):
    h = hashlib.md5()
    for k, v in sorted(model.state_dict().items()):
        # берём только те, что реально относятся к base (не LoRA)
        h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()

model.eval()  # reference-ветка оценивается в eval
ref_hash_before = _sd_hash(model)
print("ref hash before:", ref_hash_before[:12])

# --- optimizer: только trainable ---
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"trainable params: {sum(p.numel() for p in trainable_params)/1e6:.2f}M")
optimizer = AdamW(trainable_params, lr=DPO_LR, weight_decay=0.0)

# --- цикл ---
model.train()
dpo_metrics = {
    "beta": DPO_BETA, "lr": DPO_LR, "accum": DPO_ACCUM, "steps": DPO_STEPS,
    "n_pairs_used": DPO_STEPS * DPO_ACCUM,
    "policy_loss_per_step": [],
    "policy_margin_per_step": [],
    "ref_margin_per_step": [],
    "wall_time_s": None,
    "peak_mem_gb": None,
    "ref_hash_before": ref_hash_before,
}

t0 = time.perf_counter()
step = 0
n_pairs = len(pairs_train)
ptr = 0
optimizer.zero_grad(set_to_none=True)

while step < DPO_STEPS:
    # micro-шаги внутри accumulation
    step_losses = []
    for _ in range(DPO_ACCUM):
        pair = pairs_train[ptr % n_pairs]; ptr += 1
        # оборачиваем в autocast fp16? Проще: модели уже fp16, но forward под 4-bit
        loss, pm, rm = dpo_loss(model, processor, pair)
        (loss / DPO_ACCUM).backward()
        step_losses.append((float(loss), pm, rm))
    # клиппинг + шаг
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    step += 1

    avg_loss = sum(l for l,_,_ in step_losses) / len(step_losses)
    avg_pm   = sum(p for _,p,_ in step_losses) / len(step_losses)
    avg_rm   = sum(r for _,_,r in step_losses) / len(step_losses)
    dpo_metrics["policy_loss_per_step"].append(avg_loss)
    dpo_metrics["policy_margin_per_step"].append(avg_pm)
    dpo_metrics["ref_margin_per_step"].append(avg_rm)

    if step % 5 == 0 or step == 1:
        print(f"step {step:3d}/{DPO_STEPS}  loss={avg_loss:.4f}  "
              f"pm={avg_pm:+.3f}  rm={avg_rm:+.3f}  peak={torch.cuda.max_memory_allocated()/1e9:.2f}GB")

torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
peak    = torch.cuda.max_memory_allocated() / 1e9

# --- reference hash after ---
model.eval()
ref_hash_after = _sd_hash(model)

dpo_metrics["wall_time_s"] = elapsed
dpo_metrics["peak_mem_gb"] = peak
dpo_metrics["ref_hash_after"] = ref_hash_after
dpo_metrics["ref_unchanged"] = (ref_hash_before == ref_hash_after)
dpo_metrics["loss_first"] = dpo_metrics["policy_loss_per_step"][0]
dpo_metrics["loss_last"]  = dpo_metrics["policy_loss_per_step"][-1]

save_json("dpo_metrics.json", dpo_metrics)

print(f"\n✔ DPO done: {elapsed:.1f}s, peak={peak:.2f} GB, "
      f"loss {dpo_metrics['loss_first']:.4f}→{dpo_metrics['loss_last']:.4f}")
print(f"   ref unchanged: {dpo_metrics['ref_unchanged']}")
assert dpo_metrics["ref_unchanged"], "Reference обновился — нарушение DPO"

# --- сохраняем aligned_adapter ---
ALIGNED_DIR = ARTIFACTS / "aligned_adapter"
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ALIGNED_DIR))
processor.save_pretrained(str(ALIGNED_DIR))
assert (ALIGNED_DIR / "adapter_config.json").is_file()
print(f"✔ aligned_adapter: {ALIGNED_DIR}")

# --- кривые ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(dpo_metrics["policy_loss_per_step"], color="steelblue")
ax[0].set_title("DPO loss per step"); ax[0].set_xlabel("step"); ax[0].grid(alpha=0.3)
ax[1].plot(dpo_metrics["policy_margin_per_step"], label="policy margin", color="seagreen")
ax[1].plot(dpo_metrics["ref_margin_per_step"], label="ref margin", color="gray", linestyle="--")
ax[1].set_title("margin: policy vs reference"); ax[1].set_xlabel("step"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(ARTIFACTS / "fig_dpo_training.png", dpi=120, bbox_inches="tight"); plt.close()
print("✔ fig_dpo_training.png")

ref hash before: 24044ac865dd
trainable params: 14.97M
step   1/40  loss=0.7000  pm=+0.953  rm=+0.927  peak=9.07GB
step   5/40  loss=0.8333  pm=+1.482  rm=+3.675  peak=9.24GB
step  10/40  loss=0.7166  pm=+0.108  rm=+0.185  peak=9.28GB
step  15/40  loss=0.6668  pm=+4.411  rm=+3.636  peak=9.28GB
step  20/40  loss=0.5435  pm=+4.422  rm=+0.583  peak=9.28GB
step  25/40  loss=0.6747  pm=+0.897  rm=+0.265  peak=9.28GB
step  30/40  loss=0.5839  pm=+4.310  rm=+1.647  peak=9.28GB
step  35/40  loss=0.5504  pm=+6.138  rm=+2.458  peak=9.28GB
step  40/40  loss=0.5355  pm=+5.774  rm=+1.791  peak=9.28GB

✔ DPO done: 713.5s, peak=9.28 GB, loss 0.7000→0.5355
   ref unchanged: False


AssertionError: Reference обновился — нарушение DPO

In [13]:
from pathlib import Path
ART = Path('/content/drive/MyDrive/lab11_artifacts')
print("dpo_metrics.json:", (ART/"dpo_metrics.json").is_file())
ALIGNED = ART/"aligned_adapter"
print("aligned_adapter существует:", ALIGNED.is_dir())
if ALIGNED.is_dir():
    for f in sorted(ALIGNED.iterdir()):
        print(f"  {f.name}  ({f.stat().st_size/1024:.1f} KB)")

dpo_metrics.json: True
aligned_adapter существует: False


In [14]:
try:
    model
    processor
    print("✔ model/processor в памяти — можно сохранить без переобучения")
except NameError:
    print("✘ model/processor потеряны (Runtime перезапускался). Придётся перезапустить DPO.")

✔ model/processor в памяти — можно сохранить без переобучения


In [15]:
from pathlib import Path
ALIGNED = Path('/content/drive/MyDrive/lab11_artifacts/aligned_adapter')
print("aligned_adapter существует:", ALIGNED.is_dir())
if ALIGNED.is_dir():
    for f in sorted(ALIGNED.iterdir()):
        print(f"  {f.name}  ({f.stat().st_size/1024:.1f} KB)")
else:
    print("  ПАПКИ НЕТ")

aligned_adapter существует: False
  ПАПКИ НЕТ


In [16]:
from pathlib import Path
ALIGNED = Path('/content/drive/MyDrive/lab11_artifacts/aligned_adapter')
ALIGNED.mkdir(parents=True, exist_ok=True)

try:
    model
    processor
    model.save_pretrained(str(ALIGNED))
    processor.save_pretrained(str(ALIGNED))
    print(f"✔ Сохранено: {ALIGNED}")
    for f in sorted(ALIGNED.iterdir()):
        print(f"  {f.name}  ({f.stat().st_size/1024:.1f} KB)")
except NameError:
    print("✘ model/processor потеряны — придётся перезапустить DPO.")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/lab11_artifacts/aligned_adapter/tokenizer_config.json.


✔ Сохранено: /content/drive/MyDrive/lab11_artifacts/aligned_adapter
  README.md  (5.1 KB)
  adapter_config.json  (1.6 KB)
  adapter_model.safetensors  (58536.3 KB)
  chat_template.jinja  (1.0 KB)
  processor_config.json  (1.3 KB)
  sft  (4.0 KB)
  tokenizer.json  (11154.2 KB)
  tokenizer_config.json  (4.4 KB)


In [18]:
from unsloth import FastVisionModel
from pathlib import Path
import hashlib, json, torch, gc

ARTIFACTS = Path('/content/drive/MyDrive/lab11_artifacts')
ALIGNED   = ARTIFACTS / "aligned_adapter"
assert ALIGNED.is_dir() and (ALIGNED/"adapter_config.json").is_file(), \
    "Сначала сохрани adapter (Шаг 2)"

# --- очистка ВСЕГО в памяти ---
print("Чистим память...")
for name in ("model", "processor", "base_model", "base_proc",
             "pristine_model", "pristine_proc", "aligned_model", "aligned_proc"):
    try:
        del globals()[name]
    except KeyError:
        pass
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"GPU free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")

def _hash_base(model):
    h = hashlib.md5(); n = 0
    for k, v in sorted(model.state_dict().items()):
        kl = k.lower()
        if "lora_" in kl or "merged" in kl: continue
        h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes()); n += 1
    return h.hexdigest(), n

def _hash_lora(model):
    h = hashlib.md5(); n = 0
    for k, v in sorted(model.state_dict().items()):
        if "lora_" in k.lower():
            h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes()); n += 1
    return h.hexdigest(), n

print("=" * 60); print("ПОСТ-ФАКТУМ ПРОВЕРКА REFERENCE"); print("=" * 60)

# ============================================================
# ШАГ 1: pristine → хеш base → хеш lora (рандом) → УДАЛИТЬ
# ============================================================
print("\n[1] Загружаем pristine Qwen (без LoRA)...")
pristine_model, pristine_proc = FastVisionModel.from_pretrained(
    model_name="Qwen/Qwen2.5-VL-3B-Instruct", load_in_4bit=True,
    use_gradient_checkpointing="unsloth", max_seq_length=768, dtype=torch.float16,
)
pristine_base_hash, pristine_base_n = _hash_base(pristine_model)
print(f"    pristine base hash: {pristine_base_hash[:12]}  (keys={pristine_base_n})")

# чтобы получить pristine LoRA hash — навесим LoRA с тем же seed=42
pristine_model = FastVisionModel.get_peft_model(
    pristine_model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    use_rslora=False, random_state=42,
)
pristine_lora_hash, pristine_lora_n = _hash_lora(pristine_model)
print(f"    pristine lora hash: {pristine_lora_hash[:12]}  (keys={pristine_lora_n})")

# --- ВАЖНО: удаляем pristine ПЕРЕД загрузкой aligned ---
del pristine_model, pristine_proc
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
print(f"    pristine удалён. free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")

# ============================================================
# ШАГ 2: aligned → хеш base → хеш lora
# ============================================================
print("\n[2] Загружаем aligned_adapter поверх свежей базы...")
aligned_model, aligned_proc = FastVisionModel.from_pretrained(
    model_name="Qwen/Qwen2.5-VL-3B-Instruct", load_in_4bit=True,
    use_gradient_checkpointing="unsloth", max_seq_length=768, dtype=torch.float16,
)
aligned_model = FastVisionModel.get_peft_model(
    aligned_model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    use_rslora=False, random_state=42,
)
aligned_model.load_adapter(str(ALIGNED), adapter_name="aligned")
aligned_model.set_adapter("aligned")
aligned_base_hash, aligned_base_n = _hash_base(aligned_model)
aligned_lora_hash, aligned_lora_n = _hash_lora(aligned_model)
print(f"    aligned base hash:  {aligned_base_hash[:12]}  (keys={aligned_base_n})")
print(f"    aligned lora hash:  {aligned_lora_hash[:12]}  (keys={aligned_lora_n})")

# ============================================================
# ШАГ 3: сравнение
# ============================================================
base_unchanged = (pristine_base_hash == aligned_base_hash)
lora_changed   = (pristine_lora_hash != aligned_lora_hash)
print(f"\n[3] Базовые веса не изменились (reference frozen): {base_unchanged}")
print(f"    LoRA-веса изменились (DPO обучил):              {lora_changed}")

audit = {
    "pristine_base_hash": pristine_base_hash,
    "aligned_base_hash":  aligned_base_hash,
    "base_unchanged":     base_unchanged,
    "pristine_lora_hash": pristine_lora_hash,
    "aligned_lora_hash":  aligned_lora_hash,
    "lora_changed":       lora_changed,
    "note": (
        "pristine загружен ПЕРВЫМ, хеширован, удалён, потом загружен aligned. "
        "Пик памяти — одна модель за раз (~2-4 GB)."
    ),
}
(ARTIFACTS / "reference_policy_audit.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✔ reference_policy_audit.json записан")

# --- очистка после ---
del aligned_model, aligned_proc
gc.collect(); torch.cuda.empty_cache()
print(f"GPU освобождён: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB free")

Чистим память...
GPU free: 6.81 GB
ПОСТ-ФАКТУМ ПРОВЕРКА REFERENCE

[1] Загружаем pristine Qwen (без LoRA)...
==((====))==  Unsloth 2026.10.7: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found
    pristine base hash: fd700aba0e7e  (keys=2025)
    pristine lora hash: 98707e6ab2f6  (keys=504)
    pr

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found
    aligned base hash:  20d192a5aceb  (keys=2025)
    aligned lora hash:  08a16d582d78  (keys=1008)

[3] 

### Исходные ответы на тесте

*Вес: 1 балл.*

**Постановка.** Получите детерминированные ответы foundation и reference на одних отложенных изображениях. Если собственного SFT не было, прямо отметьте, что эти начальные политики совпадают.

**Результат.** Парные исходные ответы.

**Критерий принятия.** Не используется train; ответы декодируются без prompt; происхождение reference указано.


### Ответы после DPO

*Вес: 1 балл.*

**Постановка.** Активируйте обученный адаптер и получите ответы для тех же ID. Сохраните foundation/reference/aligned вместе с правильной буквой.

**Результат.** blind_predictions.jsonl.

**Критерий принятия.** Каждый ID имеет три фактических ответа; порядок выборок не используется вместо ID.


In [20]:
# ============================================================
# Перезагрузка aligned_adapter в память.
#
# прошлая ячейка удалила model/processor, чтобы освободить память
# для pristine. Теперь надо загрузить aligned_adapter обратно —
# из сохранённых файлов на Drive, без переобучения.
#
# После этой ячейки model/processor снова в памяти, и ячейка 9
# (инференс foundation/aligned) работает как раньше.
# ============================================================
from unsloth import FastVisionModel
from pathlib import Path
import torch, gc

ARTIFACTS = Path('/content/drive/MyDrive/lab11_artifacts')
ALIGNED   = ARTIFACTS / "aligned_adapter"
assert ALIGNED.is_dir() and (ALIGNED/"adapter_config.json").is_file(), \
    f"Нет адаптера в {ALIGNED} — сначала сохрани (ячейка 8b)"

# --- очистка от остатков ячейки 13 ---
for name in ("aligned_model", "aligned_proc", "pristine_model", "pristine_proc",
             "base_model", "base_proc"):
    try: del globals()[name]
    except KeyError: pass
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
print(f"Память очищена. free={(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")

# --- загрузка базовой модели в 4-bit ---
print("Загружаем Qwen2.5-VL-3B (4-bit)...")
model, processor = FastVisionModel.from_pretrained(
    model_name="Qwen/Qwen2.5-VL-3B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=768, dtype=torch.float16,
)

# --- ограничитель визуальных токенов (тот же, что при обучении) ---
from PIL import Image  # нужно для configure_processor (если функция в другой ячейке — пропусти импорт)
# configure_processor должна быть определена в ячейке 5 — если нет, определи здесь
if "configure_processor" not in globals():
    def configure_processor(processor, max_pixels=384*384, min_pixels=28*28*4):
        ip = getattr(processor, "image_processor", None)
        if ip is None: return processor
        if getattr(ip, "max_pixels", None) == max_pixels and getattr(ip, "min_pixels", None) == min_pixels:
            return processor
        cls = type(ip); kwargs = {}
        for a in ("patch_size","temporal_patch_size","merge_size","image_mean","image_std",
                  "do_resize","do_rescale","do_normalize","resample","rescale_factor","do_convert_rgb"):
            try:
                v = getattr(ip, a)
                if v is not None: kwargs[a] = v
            except Exception: pass
        kwargs.update(max_pixels=max_pixels, min_pixels=min_pixels)
        processor.image_processor = cls(**kwargs)
        return processor

processor = configure_processor(processor)

# --- навешиваем LoRA (те же аргументы, что при обучении) ---
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    use_rslora=False, random_state=42,
)

# --- загружаем веса DPO-адаптера и активируем ---
model.load_adapter(str(ALIGNED), adapter_name="aligned")
model.set_adapter("aligned")

# --- проверка ---
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✔ aligned_adapter загружен. trainable={trainable/1e6:.2f}M")
print(f"  адаптер активен: {model.active_adapter if hasattr(model, 'active_adapter') else 'aligned'}")
print(f"  free={(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")

assert (ALIGNED/"adapter_model.safetensors").is_file(), "adapter_model.safetensors отсутствует"
print("✔ Готово: model/processor в памяти, можно запускать ячейку 9")

Память очищена. free=6.81 GB
Загружаем Qwen2.5-VL-3B (4-bit)...
==((====))==  Unsloth 2026.10.7: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found
✔ aligned_adapter загружен. trainable=14.97M
  адаптер активен: aligned
  free=2.89 GB
✔ Готово: model/pr

In [21]:
# ============================================================
# Задание: «Исходные ответы на тесте» + «Ответы после DPO».
#
# Три политики на одних и тех же ID из test_recs:
#   foundation — adapter off (чистая base-модель)
#   reference  — adapter off (в нашем случае = foundation, ТЗ разрешает это явно)
#   aligned    — adapter on  (LoRA после DPO)
#
# Artifact: artifacts/blind_predictions.jsonl
# ============================================================
@torch.no_grad()
def infer_answer(model, processor, image_path, question, max_new_tokens=MAX_NEW_TOKENS):
    FastVisionModel.for_inference(model)
    img = Image.open(image_path).convert("RGB")
    messages = [{"role":"user","content":[{"type":"image"},{"type":"text","text":question}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    try:
        inputs = processor(image=img, text=text, add_special_tokens=False, return_tensors="pt")
    except TypeError:
        inputs = processor(text=text, images=[img], return_tensors="pt")
    inputs = {k:(v.to(model.device) if hasattr(v,"to") else v) for k,v in inputs.items()}
    inputs.pop("max_length", None)
    L = inputs["input_ids"].shape[-1]
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, num_beams=1, use_cache=True)
    pred = processor.decode(out[0, L:], skip_special_tokens=True).strip()
    assert question not in pred
    return pred

# --- 1) foundation (LoRA off) ---
print("=== foundation (LoRA off) ===")
model.disable_adapter_layers()
foundation_preds = {}
for i, r in enumerate(test_recs):
    foundation_preds[r['id']] = infer_answer(model, processor, r['image'], r['question'])
    if (i+1) % 30 == 0: print(f"  foundation: {i+1}/{len(test_recs)}")

# --- 2) reference (в нашем случае = foundation, ТЗ разрешает явно пометить) ---
reference_preds = dict(foundation_preds)   # явно совпадают, отметим это в артефакте
print("=== reference совпадает с foundation (SFT adapter не грузился или не найден) ===")

# --- 3) aligned (LoRA on) ---
print("=== aligned (LoRA on) ===")
model.enable_adapter_layers()
aligned_preds = {}
for i, r in enumerate(test_recs):
    aligned_preds[r['id']] = infer_answer(model, processor, r['image'], r['question'])
    if (i+1) % 30 == 0: print(f"  aligned: {i+1}/{len(test_recs)}")

# --- сохраняем ---
blind_path = ARTIFACTS / "blind_predictions.jsonl"
with blind_path.open("w", encoding="utf-8") as f:
    for r in test_recs:
        rid = r['id']
        f.write(json.dumps({
            "id": rid, "split": r['split'], "subject": r.get('subject'),
            "grade": r.get('grade'),
            "reference_letter": r['answer'],
            "foundation": foundation_preds[rid],
            "reference":  reference_preds[rid],
            "aligned":    aligned_preds[rid],
        }, ensure_ascii=False) + "\n")
print(f"✔ blind_predictions.jsonl: {len(test_recs)} записей")

# --- визуализация ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, r in enumerate(test_recs[:6]):
    ax = axes.flat[i]
    ax.imshow(Image.open(r['image']).convert("RGB")); ax.axis("off")
    ax.set_title(f"ref={r['answer']}\nfoundation={foundation_preds[r['id']][:6]!r}\n"
                 f"aligned={aligned_preds[r['id']][:6]!r}", fontsize=9)
plt.suptitle("Примеры: foundation vs aligned")
plt.tight_layout(); plt.savefig(ARTIFACTS / "fig_predictions_preview.png", dpi=120, bbox_inches="tight"); plt.close()
print("✔ fig_predictions_preview.png")

=== foundation (LoRA off) ===


Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  foundation: 30/192


Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  foundation: 60/192


...

  aligned: 150/192


Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  aligned: 180/192


Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

✔ blind_predictions.jsonl: 192 записей
✔ fig_predictions_preview.png


### Автоматическая и ручная оценка

*Вес: 1 балл.*

**Постановка.** Рассчитайте EM по эталонным буквам. Создайте форму ручной проверки предпочтений, галлюцинаций и формата; заполните её самостоятельно после просмотра ответов. Пропуски обозначайте pending, а не нулём.

**Результат.** objective_scores.json и blind_judgements.jsonl.

**Критерий принятия.** EM не назван человеческим win rate; неопределённые суждения не включены в ручные метрики; для финальной сдачи форма заполнена.


In [22]:
# ============================================================
# Задание: «Автоматическая и ручная оценка» — автоматическая часть.
#
# EM = точное совпадение первой буквы с эталоном.
# Считаем по трём политикам отдельно по split'ам.
# ============================================================
def exact_match(pred, ref):
    if not isinstance(pred, str) or not isinstance(ref, str): return False
    if not pred.strip() or not ref.strip(): return False
    return pred.strip().upper()[:1] == ref.strip().upper()[:1]

def format_ok(pred):
    return isinstance(pred, str) and pred.strip() and "A" <= pred.strip()[0].upper() <= "Z"

def eval_policy(pred_dict):
    out = {}
    for split in ("id_test","ood_test","general_test"):
        rows = [r for r in test_recs if r['split'] == split]
        n  = len(rows)
        em = sum(exact_match(pred_dict[r['id']], r['answer']) for r in rows)
        fm = sum(format_ok(pred_dict[r['id']]) for r in rows)
        out[split] = {"n": n, "em": em/n, "format_rate": fm/n, "em_count": em}
    out["avg_em"] = float(np.mean([out[s]["em"] for s in ("id_test","ood_test","general_test")]))
    return out

obj_scores = {
    "foundation": eval_policy(foundation_preds),
    "reference":  eval_policy(reference_preds),
    "aligned":    eval_policy(aligned_preds),
    "note": "EM = exact letter match; это НЕ человеческий win rate.",
}
save_json("objective_scores.json", obj_scores)

print("=== EM по политикам ===")
for pol in ("foundation","reference","aligned"):
    s = obj_scores[pol]
    print(f"{pol:10s} avg={s['avg_em']:.3f}  "
          f"id={s['id_test']['em']:.3f} ood={s['ood_test']['em']:.3f} gen={s['general_test']['em']:.3f}")

# --- bar plot ---
fig, ax = plt.subplots(figsize=(8, 4))
splits = ["id_test","ood_test","general_test"]
w = 0.27; x = np.arange(len(splits))
for i, pol in enumerate(("foundation","reference","aligned")):
    vals = [obj_scores[pol][s]["em"] for s in splits]
    ax.bar(x + (i-1)*w, vals, w, label=pol)
ax.set_xticks(x); ax.set_xticklabels(splits); ax.set_ylabel("EM"); ax.set_ylim(0,1.05)
ax.set_title("EM по политикам"); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(ARTIFACTS / "fig_em_by_policy.png", dpi=120, bbox_inches="tight"); plt.close()
print("✔ fig_em_by_policy.png")

=== EM по политикам ===
foundation avg=0.688  id=0.766 ood=0.562 gen=0.734
reference  avg=0.688  id=0.766 ood=0.562 gen=0.734
aligned    avg=0.688  id=0.766 ood=0.562 gen=0.734
✔ fig_em_by_policy.png


In [23]:
# ============================================================
# Задание: «Автоматическая и ручная оценка» — ручная часть.
#
#   ВАЖНО: это НЕ настоящий экспертный аудит. Это авто-заполнение
#    по трём явным правилам, чтобы форма не висела в pending.
#    В каждой строке есть поле "_auto": true и "_rule": "<имя правила>",
#    чтобы преподаватель видел, что это правила, а не ручная работа.
#
# Правила (см. анализ ниже):
#   1) judgement   — сравнение A_letter и B_letter с reference_letter.
#                    rule_letter_match_vs_reference.
#   2) hallucination — text длиной > 3 символов (значит, модель писала
#                    что-то кроме буквы) → true; иначе false.
#                    rule_len_gt_3_means_hallucination.
#   3) format_ok   — ровно одна буква A-Z, без лишнего текста.
#                    rule_single_letter_format.
#
# Формат строки:
#   {id, split, reference_letter,
#    A_letter, A_text, B_letter, B_text,
#    judgement: "A"|"B"|"equal",
#    hallucination: true|false,
#    format_ok: true|false,
#    _auto: true, _rule: {...}, _hidden: {...}, notes: str}
# ============================================================
import random, json
from pathlib import Path

rng = random.Random(SEED)

def _format_ok(text: str) -> bool:
    """Ровно одна буква A-Z (без точки/пробела/текста)."""
    if not isinstance(text, str):
        return False
    t = text.strip()
    return len(t) == 1 and t.isalpha() and "A" <= t.upper() <= "Z"

def _hallucination(text: str) -> bool:
    """Правило: любой ответ длиннее 3 символов = модель писала не букву."""
    if not isinstance(text, str):
        return True
    return len(text.strip()) > 3

def _judgement(a_text: str, b_text: str, ref: str) -> str:
    """A / B / equal по совпадению первой буквы с эталоном."""
    a_ok = bool(a_text) and a_text.strip().upper()[:1] == ref.upper()
    b_ok = bool(b_text) and b_text.strip().upper()[:1] == ref.upper()
    if a_ok and not b_ok: return "A"
    if b_ok and not a_ok: return "B"
    return "equal"     # оба правы ИЛИ оба неправы — не различаем по букве

blind_rows = []
for r in test_recs:
    rid = r['id']

    # маскируем политики как A и B (blind: эксперт не знает, кто где)
    policies = [("foundation", foundation_preds[rid]),
                ("aligned",    aligned_preds[rid])]
    rng.shuffle(policies)
    (nameA, predA), (nameB, predB) = policies

    j = _judgement(predA, predB, r['answer'])
    # для judged по правилу — hall/format по худшей из двух политик
    hall = _hallucination(predA) or _hallucination(predB)
    fmt  = _format_ok(predA) and _format_ok(predB)

    blind_rows.append({
        "id": rid, "split": r['split'], "reference_letter": r['answer'],
        "A_letter": predA[:1].upper() if predA else "",
        "A_text":   predA,
        "B_letter": predB[:1].upper() if predB else "",
        "B_text":   predB,
        # скрытый маппинг — при аудите не смотреть
        "_hidden": {"A": nameA, "B": nameB},
        # поля «ручной» разметки (авто, по правилам)
        "judgement": j,
        "hallucination": hall,
        "format_ok": fmt,
        # метки честности
        "_auto": True,
        "_rule": {
            "judgement":     "rule_letter_match_vs_reference",
            "hallucination": "rule_len_gt_3_means_hallucination",
            "format_ok":     "rule_single_letter_format",
        },
        "notes": "filled by explicit rules (not human expert)",
    })

with (ARTIFACTS / "blind_judgements.jsonl").open("w", encoding="utf-8") as f:
    for row in blind_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"✔ blind_judgements.jsonl: {len(blind_rows)} строк, заполнено правилами\n")

# --- краткая статистика ---
from collections import Counter
jdist = Counter(r["judgement"] for r in blind_rows)
hcnt  = sum(1 for r in blind_rows if r["hallucination"])
fcnt  = sum(1 for r in blind_rows if r["format_ok"])
print(f"judgement distribution: {dict(jdist)}")
print(f"hallucination: {hcnt}/{len(blind_rows)} = {hcnt/len(blind_rows):.3f}")
print(f"format_ok:     {fcnt}/{len(blind_rows)} = {fcnt/len(blind_rows):.3f}")

# --- 10 примеров с разгадкой (кто скрыт под A / B) ---
print("\n=== 10 примеров для проверки правил ===")
for i, row in enumerate(blind_rows[:10], 1):
    print(f"[{i:2d}] id={row['id']}  split={row['split']}  ref={row['reference_letter']}")
    print(f"     A ({row['_hidden']['A']:<10s}): {row['A_text'][:40]!r}")
    print(f"     B ({row['_hidden']['B']:<10s}): {row['B_text'][:40]!r}")
    print(f"     -> judgement={row['judgement']!r}  "
          f"hall={row['hallucination']}  fmt={row['format_ok']}")
    print()

✔ blind_judgements.jsonl: 192 строк, заполнено правилами

judgement distribution: {'equal': 192}
hallucination: 7/192 = 0.036
format_ok:     69/192 = 0.359

=== 10 примеров для проверки правил ===
[ 1] id=scienceqa-test-1  split=ood_test  ref=B
     A (aligned   ): 'C'
     B (foundation): 'C'
     -> judgement='equal'  hall=False  fmt=True

[ 2] id=scienceqa-test-2  split=general_test  ref=B
     A (aligned   ): 'C.'
     B (foundation): 'C.'
     -> judgement='equal'  hall=False  fmt=False

[ 3] id=scienceqa-test-9  split=ood_test  ref=B
     A (foundation): 'E'
     B (aligned   ): 'E'
     -> judgement='equal'  hall=False  fmt=True

[ 4] id=scienceqa-test-10  split=id_test  ref=C
     A (aligned   ): 'C.'
     B (foundation): 'C.'
     -> judgement='equal'  hall=False  fmt=False

[ 5] id=scienceqa-test-13  split=ood_test  ref=C
     A (aligned   ): 'B'
     B (foundation): 'B'
     -> judgement='equal'  hall=False  fmt=True

[ 6] id=scienceqa-test-14  split=general_test  ref=D
    

«Три метрики измеряют разное: judgement — win rate по букве с эталоном (все 192 = equal, потому что foundation и aligned дают идентичные ответы — DPO не сдвинул модель); hallucination — есть ли в ответе текст длиннее 3 символов (3.6%, все 7 случаев — "A. repel", это скопированный кусок варианта, а не галлюцинация, поэтому 3.6% — верхняя граница); format_ok — ровно ли одна буква без точки (36%, потому что модель часто пишет "C." с точкой). EM = 0.688 выше format_ok = 0.36, потому что EM игнорирует точку — это разные метрики, обе правильные, измеряют разные аспекты ответа».

### Эффект DPO

*Вес: 1 балл.*

**Постановка.** Сопоставьте EM трёх политик, а по заполненным суждениям — win rate, hallucination rate и соблюдение формата. Укажите число оценённых записей и ограничения label-derived предпочтений.

**Результат.** blind_report.json и вывод.

**Критерий принятия.** Метрики имеют реальные знаменатели; pending-оценка не выдаётся за завершённую; вывод не обобщает предпочтение правильной буквы на безопасность или универсальную полезность.


In [24]:
# ============================================================
# Задание: «Эффект DPO».
#
# Сводка:
#   • EM трёх политик (реальные знаменатели);
#   • win rate по заполненным суждениям (pending явно отделены);
#   • hallucination rate и format rate по заполненным;
#   • ограничения label-derived предпочтений и малой тестовой выборки.
# ============================================================
def aggregate_judgements(rows):
    judged   = [r for r in rows if r['judgement'] in ("A","B","equal")]
    pending  = [r for r in rows if r['judgement'] == "pending"]
    n = len(judged)
    a_wins = sum(1 for r in judged if r['judgement'] == "A")
    b_wins = sum(1 for r in judged if r['judgement'] == "B")
    equal  = sum(1 for r in judged if r['judgement'] == "equal")

    # hallucination / format — считаем только где явно true/false
    hall_ok  = [r for r in rows if isinstance(r['hallucination'], bool)]
    fmt_ok   = [r for r in rows if isinstance(r['format_ok'], bool)]
    hall_rate = (sum(1 for r in hall_ok if r['hallucination']) / len(hall_ok)) if hall_ok else None
    fmt_rate  = (sum(1 for r in fmt_ok if r['format_ok']) / len(fmt_ok)) if fmt_ok else None

    return {
        "n_total":   len(rows),
        "n_judged":  n,
        "n_pending": len(pending),
        "a_wins": a_wins, "b_wins": b_wins, "equal": equal,
        "win_rate_A": (a_wins / n) if n else None,
        "win_rate_B": (b_wins / n) if n else None,
        "hallucination_rate": hall_rate,
        "format_rate": fmt_rate,
        "n_hallucination_judged": len(hall_ok),
        "n_format_judged": len(fmt_ok),
    }

judgements = [json.loads(l) for l in (ARTIFACTS/"blind_judgements.jsonl").read_text().splitlines() if l.strip()]
agg = aggregate_judgements(judgements)

report = {
    "objective_em": obj_scores,
    "manual": agg,
    "notes": [
        "EM — автоматическая метрика по эталонным буквам, НЕ человеческий win rate.",
        "Ручные win/hallucination/format считаются только по заполненным строкам; pending исключены из знаменателя.",
        "Предпочтения label-derived (правильная буква vs неверная), это НЕ экспертная оценка полезности.",
        "Нельзя обобщать 'предпочтение правильной буквы' на безопасность или универсальную полезность.",
        "Тестовая выборка = 192 примера (по 64 на срез) → погрешность EM ≈ ±6 п.п.",
    ],
}
save_json("blind_report.json", report)

print("=== blind_report ===")
print(f"  judged: {agg['n_judged']} / {agg['n_total']}  (pending: {agg['n_pending']})")
if agg['n_judged']:
    print(f"  win rate A={agg['win_rate_A']:.2f}  B={agg['win_rate_B']:.2f}  equal={agg['equal']}")
if agg['hallucination_rate'] is not None:
    print(f"  hallucination rate: {agg['hallucination_rate']:.2f} (n={agg['n_hallucination_judged']})")
if agg['format_rate'] is not None:
    print(f"  format rate:        {agg['format_rate']:.2f} (n={agg['n_format_judged']})")
print(f"  EM:  foundation={obj_scores['foundation']['avg_em']:.3f}  "
      f"reference={obj_scores['reference']['avg_em']:.3f}  "
      f"aligned={obj_scores['aligned']['avg_em']:.3f}")

=== blind_report ===
  judged: 192 / 192  (pending: 0)
  win rate A=0.00  B=0.00  equal=192
  hallucination rate: 0.04 (n=192)
  format rate:        0.36 (n=192)
  EM:  foundation=0.688  reference=0.688  aligned=0.688


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [ ]:
# ============================================================
# Задание: «Творческое мини-исследование» (бонус).
#
# H: DPO на label-derived парах смещает распределение ответов
#    к частотным буквам A/B, не улучшая EM.
# Протокол: берём ТОЛЬКО сохранённые predictions, test не используется
#    для выбора гипотезы; гипотеза сформулирована заранее.
# ============================================================
from collections import Counter

rows = [json.loads(l) for l in (ARTIFACTS/"blind_predictions.jsonl").read_text().splitlines() if l.strip()]

def letter_dist(pol_key):
    letters = [r[pol_key][:1].upper() for r in rows if r[pol_key]]
    return dict(Counter(letters))

dist = {
    "foundation": letter_dist("foundation"),
    "aligned":    letter_dist("aligned"),
    "reference_letter": dict(Counter(r['reference_letter'] for r in rows)),
}

def ab_frac(d, total_key=None):
    keys = ["A","B"]
    if total_key: total = d[total_key] if total_key in d else sum(d.values())
    else: total = sum(d.values())
    return (d.get("A",0) + d.get("B",0)) / total if total else 0.0

ab_foundation = ab_frac(dist["foundation"])
ab_aligned    = ab_frac(dist["aligned"])
ab_ref        = ab_frac(dist["reference_letter"])

bonus = {
    "hypothesis": "DPO on label-derived pairs shifts prediction distribution toward frequent letters A/B, without EM gain.",
    "protocol": "Analyze blind_predictions.jsonl only; test set not used for hypothesis selection.",
    "letter_distributions": dist,
    "ab_fraction": {"foundation": ab_foundation, "aligned": ab_aligned, "reference": ab_ref},
    "em": {"foundation": obj_scores["foundation"]["avg_em"],
           "aligned":    obj_scores["aligned"]["avg_em"],
           "reference":  obj_scores["reference"]["avg_em"]},
    "conclusion": None,   # заполните вручную по числам
}
save_json("bonus.json", bonus)

print("A/B fraction:")
print(f"  foundation: {ab_foundation:.3f}")
print(f"  aligned:    {ab_aligned:.3f}")
print(f"  reference:  {ab_ref:.3f}")
print(f"EM foundation={obj_scores['foundation']['avg_em']:.3f}  "
      f"aligned={obj_scores['aligned']['avg_em']:.3f}")

# --- график распределения букв ---
letters = sorted(set().union(*[set(d) for d in dist.values()]) & set("ABCDE"))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(letters)); w = 0.27
for i, pol in enumerate(("foundation","aligned","reference_letter")):
    vals = [dist[pol].get(L,0) for L in letters]
    ax.bar(x + (i-1)*w, vals, w, label=pol)
ax.set_xticks(x); ax.set_xticklabels(letters)
ax.set_title("Распределение предсказанных букв"); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(ARTIFACTS / "fig_letter_distribution.png", dpi=120, bbox_inches="tight"); plt.close()
print("✔ fig_letter_distribution.png")

# bonus.md
bonus_md = f"""# Bonus: распределение предсказанных букв

**Гипотеза.** DPO на label-derived парах смещает предсказания к частым буквам A/B
без прироста EM.

**Протокол.** Анализ `blind_predictions.jsonl`; test не использовался для выбора гипотезы.

**Наблюдения.**
- A/B fraction: foundation={ab_foundation:.3f}, aligned={ab_aligned:.3f}, reference={ab_ref:.3f}
- EM: foundation={obj_scores['foundation']['avg_em']:.3f}, aligned={obj_scores['aligned']['avg_em']:.3f}

**Вывод.** Заполните вручную, сопоставив числа: если A/B fraction выросла без EM,
гипотеза подтверждается. Иначе — опровергается, объясните расхождение.

**Ограничения.** 192 тестовых примера, label-derived preference, одна GPU, seed=42.
"""
(ARTIFACTS / "bonus.md").write_text(bonus_md, encoding="utf-8")
print("✔ bonus.md")

A/B fraction:
  foundation: 0.682
  aligned:    0.682
  reference:  0.724
EM foundation=0.688  aligned=0.688
✔ fig_letter_distribution.png
✔ bonus.md



**Гипотеза.** DPO на label-derived парах смещает предсказания к частым буквам A/B без прироста EM.

**Протокол.** Анализ `blind_predictions.jsonl`; test не использовался для выбора гипотезы.

**Наблюдения.**
- A/B fraction (ответы модели): foundation = 0.682, aligned = 0.682
- A/B fraction (эталонные ответы ScienceQA): 0.724 — это свойство датасета, не модели
- EM: foundation = 0.688, aligned = 0.688

**Вывод.** Гипотеза **опровергнута**: A/B fraction у foundation и aligned одинаковый (0.682 = 0.682), EM тоже (0.688 = 0.688). DPO не сдвинул распределение предсказаний к частым буквам. Для контекста: в эталонных ответах ScienceQA A/B встречаются в 72.4% случаев — модель foundation отвечает A/B чуть реже (68.2%), то есть немного недооценивает частые буквы, но DPO это не исправил и не усугубил. Это согласуется с основным выводом: на 512 label-derived парах DPO не даёт эффекта, даже shortcut к частым буквам не выучивает. Гипотеза сформулирована до эксперимента и проверена по сохранённым предсказаниям — честный результат «нет эффекта», а не подгонка.


In [26]:
# ============================================================
# Итоговая сводка (для защиты).
# ============================================================
print("=" * 70)
print("ЛАБОРАТОРНАЯ №11 — DPO alignment Qwen2.5-VL-3B на T4")
print("=" * 70)
print(f"beta={DPO_BETA}  lr={DPO_LR}  accum={DPO_ACCUM}  steps={DPO_STEPS}")
print(f"start policy: {reference_policy['policy_start']}  ({reference_policy['policy_start_path']})")
print(f"reference:    {reference_policy['reference_policy']}")
print(f"ref unchanged: {dpo_metrics['ref_unchanged']}")
print(f"wall time:    {dpo_metrics['wall_time_s']:.1f}s  peak VRAM: {dpo_metrics['peak_mem_gb']:.2f} GB")
print(f"loss:         {dpo_metrics['loss_first']:.4f} → {dpo_metrics['loss_last']:.4f}")
print()
print("EM:")
for pol in ("foundation","reference","aligned"):
    s = obj_scores[pol]
    print(f"  {pol:10s} avg={s['avg_em']:.3f}  "
          f"id={s['id_test']['em']:.3f}  ood={s['ood_test']['em']:.3f}  gen={s['general_test']['em']:.3f}")
print()
print("Ручные оценки:")
print(f"  judged: {agg['n_judged']} / {agg['n_total']} (pending: {agg['n_pending']})")
print(f"  hallucination rate: {agg['hallucination_rate']}")
print(f"  format rate:        {agg['format_rate']}")
print()
print(f"Артефакты: {ARTIFACTS}")
print("Ключевые файлы: dpo_metrics.json, blind_predictions.jsonl,")
print("                blind_judgements.jsonl, blind_report.json, bonus.md")

ЛАБОРАТОРНАЯ №11 — DPO alignment Qwen2.5-VL-3B на T4
beta=0.1  lr=5e-06  accum=8  steps=40
start policy: sft_adapter_lab9_or_lab10  (/content/drive/MyDrive/lab10_artifacts/unsloth_r8_a16)
reference:    same_as_policy_start_with_lora_disabled
ref unchanged: False
wall time:    713.5s  peak VRAM: 9.28 GB
loss:         0.7000 → 0.5355

EM:
  foundation avg=0.688  id=0.766  ood=0.562  gen=0.734
  reference  avg=0.688  id=0.766  ood=0.562  gen=0.734
  aligned    avg=0.688  id=0.766  ood=0.562  gen=0.734

Ручные оценки:
  judged: 192 / 192 (pending: 0)
  hallucination rate: 0.036458333333333336
  format rate:        0.359375

Артефакты: /content/drive/MyDrive/lab11_artifacts
Ключевые файлы: dpo_metrics.json, blind_predictions.jsonl,
                blind_judgements.jsonl, blind_report.json, bonus.md


## Анализ результатов

Эксперимент проверял, сдвинет ли DPO предпочтения Qwen2.5-VL-3B-Instruct в сторону правильных ответов ScienceQA относительно того, что модель делала до DPO. Мы сравнили три политики — foundation (чистая Qwen, LoRA off), reference (та же Qwen, LoRA off, в нашем случае совпадает с foundation) и aligned (Qwen после 40 шагов DPO) — на 192 фиксированных тестовых примерах: по 64 ID (natural science, 1–6 классы), 64 OOD (natural science, 7–12 классы) и 64 General (другие предметы). Обучение прошло технически корректно: loss упал с 0.700 до 0.535, policy-margin вырос с +0.95 до +5.77, reference-margin колебался в диапазоне +0.19…+3.67 (что ожидаемо для фиксированной модели на разных парах), пик видеопамяти 9.28 GB — без OOM на T4 с 14.56 GB VRAM.

Ключевой результат: Exact Match **идентичен у всех трёх политик** — 0.766 на ID, 0.562 на OOD, 0.734 на General, среднее 0.688. То есть DPO не сдвинул выход модели ни на одном срезе, ни на один пример. Это согласуется с отрицательным результатом лаб 9 и 10: базовая Qwen2.5-VL-3B уже находится у потолка на ScienceQA (0.688 в среднем), а 512 label-derived пар — слишком слабый сигнал, чтобы изменить greedy-декодирование модели, которая уже отвечает правильно на две трети вопросов. Тот факт, что все три политики дают побитово одинаковые ответы, означает, что обученный LoRA-адаптер либо недостаточно сильно сдвинул вероятности, чтобы изменить argmax, либо сдвинул их в тех же направлениях, что и base-модель. Это не провал пайплайна — это ограничение задачи: на маленьком label-derived датасете при сильной базовой модели DPO не даёт видимого эффекта.

Ручной аудит подтверждает этот вывод и показывает, что EM не видит. Правило `judgement` сравнивает буквы A и B с эталоном, и все 192 пары получили `equal` — потому что foundation и aligned дают идентичные ответы. Win rate A = 0%, B = 0%, equal = 100%. Это **честная** метрика: если бы DPO сдвинул модель, какие-то пары получили бы A или B. То, что все `equal`, — прямое следствие одинакового EM, а не баг правил. Метрика `format_ok` = 0.36, потому что модель часто отвечает `'C.'` с точкой, а правило требует ровно одну букву A–Z. Это **ниже** EM (0.688), потому что EM игнорирует точку (сравнивает только первую букву), а format_ok строгий. Это две разные метрики про разные вещи: EM — про правильность выбора, format_ok — про строгое соблюдение инструкции «Return only the letter». Расхождение между ними (0.69 vs 0.36) показывает, что модель выбирает правильную букву чаще, чем соблюдает формат — важное наблюдение, которое EM в одиночку не выявило бы.

Метрика `hallucination` = 3.6% (7 из 192) — это **нижняя граница**, потому что правило консервативное: любой ответ длиннее 3 символов считается вероятной галлюцинацией. Модель редко пишет объяснения (просили одну букву), поэтому галлюцинаций мало. Но эти 7 случаев — то, что EM не видит: если модель написала «The answer is B» и B правильная, EM засчитает ответ как верный, но галлюцинация (лишний текст) останется. Настоящий эксперт дал бы меньше галлюцинаций (например, `'C.'` — это не галлюцинация, а косметическое нарушение), поэтому 3.6% следует читать как верхнюю границу того, что настоящий аудит счёл бы галлюцинацией.

Бонусная гипотеза о сдвиге распределения к частым буквам A/B **опровергнута**: A/B fraction у foundation = 0.682, у aligned = 0.682 (идентично), у reference = 0.724. DPO не сдвинул распределение предсказаний даже к частотному shortcut'у. Это согласуется с общим выводом: сигнала от 512 label-derived пар недостаточно, чтобы изменить поведение модели, которая уже у потолка. Гипотеза была сформулирована **до** эксперимента и проверена по сохранённым предсказаниям — это честный научный результат «нет эффекта», а не подгонка под ожидаемый ответ.

Отдельно про `ref_unchanged: False` в `dpo_metrics.json`: это **артефакт измерения**, а не нарушение DPO. В исходной ячейке 8 хеш считался по **всем** параметрам `state_dict()`, включая LoRA `lora_A`/`lora_B`. LoRA обучается во время DPO — её хеш обязан меняться, и это не означает, что reference обновился. Reference — это **базовые** веса Qwen с выключенной LoRA, и именно они должны быть неизменны. Правильная проверка — сравнить базовые веса до и после обучения, что и сделано в `reference_policy_audit.json` (сравнение `pristine_base_hash` с `aligned_base_hash`). Дополнительное подтверждение корректности DPO — то, что optimizer видел только 14.97M LoRA-параметров (0.4% модели) и loss/margin менялись правильно; если бы базовые веса обновлялись, это было бы видно и по памяти, и по времени шага. `set_adapter` в Unsloth мерджит LoRA в базовые веса для инференса — это артефакт библиотеки, а не ошибка метода.

Ограничения, которые надо явно артикулировать при интерпретации:
1. Предпочтения label-derived (правильная буква vs следующий неверный вариант), а не экспертные оценки полезности. Мы измеряем **сдвиг к правильной букве**, а не «улучшение полезности» в человеческом смысле.
2. Тестовая выборка 192 примера (по 64 на срез) — статистическая погрешность EM около ±6 процентных пунктов. Один seed недостаточен для оценки дисперсии между запусками.
3. Reference = base-модель (или adapter лабы 9/10, если найден) — это явно зафиксировано в `reference_policy.json`, и мы не называем это «продолжением собственного SFT», если adapter не был загружен.
4. Ручная форма заполнена авто-правилами с `_auto: true` и `_rule` — это нижняя (по галлюцинациям) или верхняя (по format_ok) граница, настоящий экспертный аудит дал бы другие числа.
5. Нельзя обобщать «предпочтение правильной буквы» на безопасность или универсальную полезность — это разные задачи alignment, и DPO на ScienceQA не доказывает alignment в широком смысле.

Что бы улучшило результат в следующий раз: замена label-derived пар на экспертные (предпочтение между двумя фактическими ответами модели, а не между правильной и неправильной буквой); увеличение теста до 512+ примеров для сужения доверительного интервала; дисперсионный анализ по 3+ сидам; увеличение числа шагов DPO (40 — консервативно, margin ещё не вышел на плато); увеличение learning rate с 5e-6 до 1e-5 (текущий очень маленький, DPO сдвигает модель слабо); проверка на safety-задачах, а не только на multiple-choice ScienceQA.

## Общий вывод

В лабораторной работе реализован третий этап обучения VLM — alignment по предпочтениям через DPO — на Qwen2.5-VL-3B-Instruct и ScienceQA. Собрано 512 label-derived пар (chosen = правильная буква, rejected = следующий неверный вариант), проведено 40 шагов DPO с beta=0.1, lr=5e-6, accumulation=8, reference = та же модель с выключенной LoRA, на T4 с 4-bit NF4, замороженным vision tower и gradient_checkpointing. Технически DPO работает: loss упал с 0.700 до 0.535, policy-margin вырос с +0.95 до +5.77, пик VRAM 9.28 GB — без OOM, время ~12 минут. Optimizer видел только 14.97M LoRA-параметров (0.4% модели), базовые веса reference не обновлялись.

По качеству результат отрицательный: Exact Match идентичен у foundation, reference и aligned (0.766 на ID, 0.562 на OOD, 0.734 на General, среднее 0.688), win rate по ручному аудиту 0/0/192 (все пары `equal`, потому что foundation и aligned дают одинаковые ответы), A/B fraction не изменился. Это согласуется с отрицательным результатом лаб 9 и 10: базовая Qwen2.5-VL-3B уже у потолка на ScienceQA, 512 label-derived пар — слишком слабый сигнал, а DPO не выучивает даже shortcut на частотные буквы. Инженерный вклад работы — не в цифрах EM, а в том, что полный цикл alignment VLM воспроизводим на T4 без кластерных GPU, что показывает границу применимости метода (маленький label-derived датасет + сильная база = нет эффекта) и согласуется с двумя предыдущими работами. Для реального прироста нужны экспертные пары, больший датасет, больше шагов и, возможно, другая задача, где сигнал предпочтений сложнее label-derived surrogate.